In [15]:
from langgraph.graph import StateGraph , START , END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import BaseMessage , HumanMessage
from typing import TypedDict , Annotated
from dotenv import load_dotenv
load_dotenv()

True

#### State

In [16]:
class ChatState(TypedDict):
    messages : Annotated[
        list[BaseMessage],
        add_messages,
    ]

#### LLMs

In [17]:
model = ChatOpenRouter(
    model = 'openrouter/free'
)

#### Node Functions

In [18]:
def chat_node(state:ChatState):
    
    messages = state["messages"]

    response = model.invoke(messages)

    return {
        "messages" : [response]
    }

#### Presistence Memory

In [19]:
checkpointer = MemorySaver()

#### Graph

In [20]:
graph = StateGraph(ChatState)

# Node
graph.add_node("chat_node" , chat_node)


# Edges
graph.add_edge(START , "chat_node")
graph.add_edge("chat_node" , END)

# Compile
chatbot = graph.compile(checkpointer=checkpointer)

In [21]:
initial_state = {
    "messages" : [
        HumanMessage(
            content = "what is the capital of india"
        )
    ]
}

final_state = chatbot.invoke(
    initial_state,
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

In [22]:
final_state["messages"][-1].content

'\n\nThe capital of India is **New Delhi**.\n\nIt is a part of the larger National Capital Territory (NCT) of Delhi. New Delhi is the seat of all three branches of the Government of India: the Executive, the Legislature, and the Judiciary.'

In [23]:
thread_id = '1'
while True:
    user_message = input("User: ")
    print("User:" , user_message)
    if user_message.strip().lower() in ['exit' , 'quit']:
        break

    message = {
        "messages" : HumanMessage(
            content = user_message
        )
    }

    config = {
        "configurable" : {
            "thread_id" : thread_id
        }
    }

    response = chatbot.invoke(
        message,
        config=config
    )

    result = response["messages"][-1].content

    print("ChatBot:",result)




User: Hii
ChatBot: Hello! How can I help you today?
User: I'm Sumit wsuuppp
ChatBot: Hi Sumit! Nice to meet you. 😊

How are you doing today? Is there anything specific you'd like to chat about or any questions I can help you with?
User: can you tell me how can i learn LangSmith in short
ChatBot: 

Of course! Learning LangSmith effectively in a short time is all about focusing on the core purpose: **debugging and evaluating your LLM applications.**

Here is a rapid, step-by-step guide to get you started.

### The "In a Nutshell" Summary

LangSmith is a platform for **observing, debugging, and evaluating** your LangChain applications. Instead of guessing why your LLM app failed, you use LangSmith to see exactly what happened, step-by-step.

---

### Step 1: The Core Concept (5 minutes)

1.  **Sign Up:** Go to [https://www.langchain.com/langsmith](https://www.langchain.com/langsmith) and create a free account. You'll get a limited number of free traces.
2.  **What is a "Trace"?** Think of

In [24]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='what is the capital of india', additional_kwargs={}, response_metadata={}, id='7c0c7137-262d-428c-9feb-672d56aaf57f'), AIMessage(content='\n\nThe capital of India is **New Delhi**.\n\nIt is a part of the larger National Capital Territory (NCT) of Delhi. New Delhi is the seat of all three branches of the Government of India: the Executive, the Legislature, and the Judiciary.', additional_kwargs={'reasoning_content': "Hmm, the user is asking a straightforward factual question about the capital of India. This is a simple query with a clear answer. \n\nI recall that New Delhi is the capital, but I should also mention it's part of the larger National Capital Territory of Delhi for completeness. The response should be concise but include a brief context about its significance to add value without overcomplicating it.\n\nSince the question is basic, no need for lengthy explanations. Just state the answer clearly and optionally provide a